# Multimodal PDF embedding*<br><br>

Handling PDF files, which often contain a mix of text, images, and layout information, presents a challenge for traditional embedding methods. This usually requires a multimodal generative model to pre-process the documents into a format that is suitable for the embedding model. This intermediate text representations can lose critical information; for example, the structure and precise content of tables or complex layouts might not be accurately rendered
<br>

In [1]:
import os

from getpass import getpass
import nest_asyncio

from dotenv import load_dotenv

nest_asyncio.apply()

load_dotenv()
CO_API_KEY = os.environ['CO_API_KEY'] or getpass("Enter your Cohere API key: ")

In [2]:
import cohere
import numpy as np
co = cohere.Client(
    api_key=CO_API_KEY
)  # Get your free API key: https://dashboard.cohere.com/api-keys

In [3]:
from pdf2image import convert_from_path
from io import BytesIO
import base64
import chromadb
import cohere


Embed v5 solves this problem as it is designed to natively understand mixed-modality inputs. Embed v5 can directly process the PDF content, including text and images, in a single step. It generates a unified embedding that captures the semantic meaning derived from both the textual and visual elements.>

Next, generate the embeddings for these pages and store them in a vector database (in this example, we use Chroma).

In [4]:
import base64
import io
import time
import cohere
from pdf2image import convert_from_path

# 1. Initialize modern Cohere V2 Client
co = cohere.ClientV2(api_key=CO_API_KEY)

pdf_path = "files/FullManual_02_ENG.pdf"

print("🔄 Converting PDF pages to optimized images...")
# Lowering DPI slightly minimizes the image token blueprint passed to Cohere
pages = convert_from_path(pdf_path, dpi=100, thread_count=4)
total_pages = len(pages)
print(f"Total pages rendered: {total_pages}")

encoded_images = []

print("🖼️ Encoding pages to Base64 Data URIs...")
for page in pages:
    buffer = io.BytesIO()
    page.save(buffer, format="PNG")
    img_bytes = buffer.getvalue()
    base64_data = base64.b64encode(img_bytes).decode("utf-8")
    data_uri = f"data:image/png;base64,{base64_data}"
    encoded_images.append(data_uri)

# 2. Resilient Batching Loop (Safe pacing for Trial Keys)
BATCH_SIZE = 3  # Small batch to control immediate token volume per request
all_embeddings = []
# Pacing delay in seconds to ensure you stay under the 100,000 tokens/minute ceiling
TRIAL_COOLDOWN = 12  

print(f"🚀 Safe embedding processing running in batches of {BATCH_SIZE}...")
for i in range(0, total_pages, BATCH_SIZE):
    batch = encoded_images[i:i + BATCH_SIZE]
    print(f" Processing batch: pages {i+1} to {min(i + BATCH_SIZE, total_pages)}...")
    
    try:
        response = co.embed(
            model="embed-v5.0-fast",
            images=batch,
            input_type="image",
            embedding_types=["float"]
        )
        batch_vectors = response.embeddings.float
        all_embeddings.extend(batch_vectors)
        
    except cohere.core.api_error.ApiError as e:
        if e.status_code == 429:
            print("⚠️ Rate limit triggered. Waiting 30 seconds to refresh the token pool...")
            time.sleep(30)
            # Retry processing the exact same batch again
            i -= BATCH_SIZE 
            continue
        else:
            print(f"❌ API Error occurred: {e}")
            break
            
    # Add a steady delay between successful iterations to satisfy the minute windows
    if i + BATCH_SIZE < total_pages:
        print(f"😴 Sleeping for {TRIAL_COOLDOWN} seconds to avoid hitting Trial rate limits...")
        time.sleep(TRIAL_COOLDOWN)

print("\n🎉 Process Complete!")
print(f"Total vectors successfully extracted: {len(all_embeddings)}")


🔄 Converting PDF pages to optimized images...
Total pages rendered: 47
🖼️ Encoding pages to Base64 Data URIs...
🚀 Safe embedding processing running in batches of 3...
 Processing batch: pages 1 to 3...
😴 Sleeping for 12 seconds to avoid hitting Trial rate limits...
 Processing batch: pages 4 to 6...
😴 Sleeping for 12 seconds to avoid hitting Trial rate limits...
 Processing batch: pages 7 to 9...
😴 Sleeping for 12 seconds to avoid hitting Trial rate limits...
 Processing batch: pages 10 to 12...
😴 Sleeping for 12 seconds to avoid hitting Trial rate limits...
 Processing batch: pages 13 to 15...
😴 Sleeping for 12 seconds to avoid hitting Trial rate limits...
 Processing batch: pages 16 to 18...
😴 Sleeping for 12 seconds to avoid hitting Trial rate limits...
 Processing batch: pages 19 to 21...
😴 Sleeping for 12 seconds to avoid hitting Trial rate limits...
 Processing batch: pages 22 to 24...
😴 Sleeping for 12 seconds to avoid hitting Trial rate limits...
 Processing batch: pages 25 to 

In [9]:
query = "Do the speakers come an BT PAIRING Mode?"
# Generate the query embedding
query_embeddings = co.embed(
    model="embed-v5.0-pro",
    input_type="search_query",
    embedding_types=["float"],
    texts=[query],
).embeddings.float
# Search the vector database
scores = np.dot(query_embeddings, np.transpose(query_embeddings))[0]

# Sort and filter documents based on scores
top_n = 4
top_doc_idxs = np.argsort(-scores)[:top_n]

# Display search results
for idx, docs_idx in enumerate(top_doc_idxs):
    print(f"Rank: {idx+1}")
    print(f"Dodocs_idxdocs_iocument")

Rank: 1
Dodocs_idxdocs_iocument
